# Graph transformers — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Graph transformers add structural bias to global attention over nodes
A graph transformer can attend globally, but structure tells it which global interactions are plausible. These examples compute attention, shortest-path bias, degree encodings, a graph token, and the contrast between biased and un-biased attention.

In [ ]:
# 1) vanilla node self-attention scores
Q=np.array([[1.,0.],[0.,1.],[1.,1.]]); K=Q.copy(); scores=Q@K.T/math.sqrt(2); a=softmax(scores[0])
plt.figure(figsize=(5,3)); plt.bar(['0','1','2'],a); plt.title('global attention from node 0')
assert np.argmax(a)==0

In [ ]:
# 2) shortest-path bias favors nearby nodes
scores=np.array([0.7,0.0,0.7]); dist=np.array([0,1,2]); biased=softmax(scores-0.5*dist)
plt.figure(figsize=(5,3)); plt.bar(['self','near','far'],biased); plt.title('attention with distance bias')
assert biased[0]>biased[2]

In [ ]:
# 3) degree encoding changes node features
X=np.array([[1.,0.],[0.,1.],[1.,1.]]); deg=np.array([1,2,1]); enc=np.c_[X,deg]
plt.figure(figsize=(5,3)); plt.imshow(enc,cmap='Blues'); plt.colorbar(); plt.title('features plus degree encoding')
assert enc.shape==(3,3) and enc[1,2]==2

In [ ]:
# 4) graph token pools global information by attention
H=np.array([[1.,0.],[0.,1.],[1.,1.]]); token=np.array([0.5,0.5]); a=softmax(H@token); g=a@H
plt.figure(figsize=(5,3)); plt.bar(['feat0','feat1'],g); plt.title('graph token readout')
assert np.allclose(g,[0.72593138,0.72593138])

In [ ]:
# 5) structural bias changes who receives mass
base=softmax([0.7,0.0,0.7]); biased=softmax(np.array([0.7,0.0,0.7])-0.5*np.array([0,1,2]))
plt.figure(figsize=(5,3)); plt.plot(base,'-o',label='no bias'); plt.plot(biased,'-o',label='with bias'); plt.legend(); plt.title('bias shifts mass toward close nodes')
assert biased[0]-biased[2] > base[0]-base[2]